# A volume as Gaussian splats, with luxar

Strategy 3 of the *Large 3D Data* session: stop shipping the voxels. luxar fits
the intensity field with a sparse set of oriented Gaussians and ships those, so
the size of what you keep follows how much structure the sample has rather than
the grid it was sampled on.

The worked example is a sunflower head from X-ray micro-CT. Everything here was
run end to end on a laptop with an NVIDIA RTX 500 (4 GB):

| | |
|---|---|
| published scan | 1371 × 1212 × 1841 uint16, **3.06 G voxels, 6.1 GB** |
| prepared for fitting | 666 × 580 × 844 uint8, 326 M voxels, 3% carrying signal |
| fit | **186,744 splats, well under an hour** |
| web scene | **5.2 MB**, opens in any WebGL2 browser |

Two things decide whether this works at all, and both are below: **the CUDA
kernels have to be compiled**, and **the background has to be exactly zero**.

## 1. Install

Fitting needs the `gsplats` extra. Viewing needs nothing.

In [ ]:
%pip install "luxar[gsplats,io]" tifffile imagecodecs ninja

## 2. Compile the CUDA kernels

luxar cannot ship these in the wheel, so a fresh install falls back to plain
PyTorch. The fallback is not a little slower, it is **orders of magnitude**
slower: on the machine above it managed about one iteration per second, against
a few thousand per minute once the kernels were built. If a fit looks broken
rather than slow, this is why.

`nvcc`, `nvvm`, `crt` and `cccl` all have to be the **same CUDA minor version as
your PyTorch**, or the build fails in confusing ways - a newer `nvvm` emits PTX
that an older `ptxas` refuses. Check `torch.version.cuda` and pin to match.

In [ ]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

In [ ]:
# Pin the toolchain to PyTorch's CUDA minor version (13.0 here - change to match).
%pip install "nvidia-cuda-nvcc==13.0.*" "nvidia-cuda-cccl==13.0.*" \
             "nvidia-nvvm==13.0.*" "nvidia-cuda-crt==13.0.*"

In [ ]:
import os, subprocess, sys
from pathlib import Path

nv = Path(sys.prefix) / "lib" / f"python3.{sys.version_info.minor}" / "site-packages" / "nvidia" / "cu13"

# The build links against -lcudart, and the wheel only ships libcudart.so.13.
(nv / "lib64").unlink(missing_ok=True)
(nv / "lib64").symlink_to(nv / "lib")
so = nv / "lib" / "libcudart.so"
so.unlink(missing_ok=True)
so.symlink_to("libcudart.so.13")

import luxar
ext = Path(luxar.__file__).parent / "gsplats" / "models" / "gsplats" / "cuda"
env = {**os.environ, "CUDA_HOME": str(nv), "PATH": f"{nv}/bin:{os.environ['PATH']}",
       "CUDA_ARCHS": "89",   # your card's compute capability; 89 = Ada
       "MAX_JOBS": "8"}
print(subprocess.run([sys.executable, "build.py"], cwd=ext, env=env,
                     capture_output=True, text=True).stdout[-600:])

## 3. Prepare the volume

**Mask the background to exactly zero before fitting.** A fit spends parameters
wherever there is signal, so a background left at a faint non-zero costs splats
and blurs the result. On the sunflower, windowing the air and the foam holder to
zero took the fraction of voxels carrying signal from 84% down to 3%, and the
whole splat budget went into the flower instead of the tub it sat in.

`tools/make-sunflower-splats.py` in this repository does exactly that for the
published scan, straight out of the zip files. For your own data the shape is:

In [ ]:
import numpy as np, tifffile, pathlib

pathlib.Path("results/luxar").mkdir(parents=True, exist_ok=True)

vol = tifffile.imread("your-volume.tif").astype(np.float32)

LO, HI = 20000.0, 60000.0          # pick these from a histogram of your own data
vol = np.clip((vol - LO) / (HI - LO), 0, 1) * 255

# Crop away the margin that is now pure zero, so no tile of the fit is wasted.
nz = np.nonzero(vol > 3)
sl = tuple(slice(int(a.min()), int(a.max()) + 1) for a in nz)
vol = vol[sl].astype(np.uint8)

print(vol.shape, f"{vol.size/1e6:.0f} Mvoxel, {100*(vol>3).mean():.1f}% carries signal")
tifffile.imwrite("results/luxar/prepared.tif", vol, compression="deflate")

## 4. Fit, build the levels, convert

Four commands. `cal` is optional - it sweeps the splat count and reports where
held-out PSNR peaks, so you can choose `--seeds` by cross-validation instead of
by guesswork. `--tiling` is what keeps a large volume inside a small card.

`convert` is the step the README's three-command summary leaves out, and it is
the one that produces something the web viewer can open.

In [ ]:
%%bash
export CUDA_HOME=$(python -c "import sys,pathlib;print(pathlib.Path(sys.prefix)/'lib'/f'python3.{sys.version_info.minor}'/'site-packages'/'nvidia'/'cu13')")
export PATH=$CUDA_HOME/bin:$PATH
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

luxar gsplat fit results/luxar/prepared.tif results/luxar/fit.gsplats.zarr \
    --preset standard --seeds 200000 -d cuda \
    --tiling uniform --tile-size 192 --overlap 24

luxar gsplat flatten results/luxar/fit.gsplats.zarr results/luxar/flat.gsplats.zarr
luxar gsplat lod     results/luxar/flat.gsplats.zarr results/luxar/scene.gsplats.zarr --recipe levels
luxar gsplat convert results/luxar/scene.gsplats.zarr results/luxar/scene.luxar.zarr --colormap gray --tone-mapping ACES

`lod --recipe` picks the streaming topology. `levels` suits zooming across
scales; `stream` is the fastest first paint for small data; `tiles`, `overview`
and `adaptive` are for progressively larger scenes.

## 5. Look at it

`luxar viewer` serves the bundled viewer and the data together.

In [ ]:
!luxar viewer --data results/luxar/scene.luxar.zarr --port 5173 --data-port 8099

## 6. Hand it to someone else

`luxar export` writes a self-contained folder - viewer, scene and a `serve.py`
that needs nothing but Python 3. That folder is the whole deliverable: unzip and
run, no install.

It is also just static files, so the same folder works on any web host that sets
CORS - see the hosting slide of the session.

In [ ]:
!luxar export results/luxar/scene.luxar.zarr -o results/luxar/live/ --overwrite
# then: cd results/luxar/live && python serve.py

## What this does and does not buy you

- It beats **all four** size problems at once: disk, RAM, graphics memory and the
  network. The other two strategies only make the data usable where it already is.
- The result is an **approximation**. Measurements belong on the voxels. Keep the
  original.
- Empty space is free, so it pays best on sparse, structured samples - and worst
  on a volume that is signal everywhere.

### Sources

- [luxar](https://github.com/royerlab/luxar) · [documentation](https://royerlab.github.io/luxar/) · [88 live demo scenes](https://demos.luxarviewer.dev)
- Sunflower: Chitwood, Quigley & Frank, [*X-ray CT Botanical Images - Set 1*](https://doi.org/10.5281/zenodo.15684909), Zenodo 2025, CC BY 4.0